<div>

# **AI agent use Tavily**

<div dir=RTL>

# **התקנות / ספריות**

In [9]:
# התקנה
!pip install -q langgraph langchain-google-genai langchain-community tavily-python
# ספריות
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
# מפתחות
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

<div dir=RTL>

# **הגדרת מודל וכלים**

In [11]:
# כלים
tools = [TavilySearchResults(max_results=3)]

# מודל
model = ChatGoogleGenerativeAI(
    model="gemini-pro-latest",
    google_api_key=os.environ["GOOGLE_API_KEY"]
).bind_tools(tools)

<div dir=RTL>

# **הגדרת גרף**

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# צומת המודל
def call_model(state: MessagesState):
    return {"messages": [model.invoke(state["messages"])]}

# בדיקה: האם המודל ביקש להפעיל כלי?
def should_continue(state: MessagesState):
    return "tools" if state["messages"][-1].tool_calls else END

# בניית הגרף
graph = StateGraph(MessagesState)
graph.add_node("model", call_model)
graph.add_node("tools", ToolNode(tools))

graph.add_edge(START, "model")
graph.add_conditional_edges("model", should_continue)
graph.add_edge("tools", "model")

agent = graph.compile()

# הצגת הגרף
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

<div dir=RTL>

# **הפעלת סוכן**

In [13]:
from langchain_core.messages import HumanMessage

# שאל את הסוכן שאלה
question = "מה החדשות הכי מעניינות בתחום ה-AI היום?"
result = agent.invoke({"messages": [HumanMessage(content=question)]})

# **הדפסת התשובה**

In [ ]:
# הדפסת תשובה
print(result["messages"][-1].content)

# **===========================**

In [ ]:
from langchain_core.messages import HumanMessage

# שאל את הסוכן שאלה
question = "מה החדשות הכי מעניינות בתחום ה-AI היום?"

messages = []
for s in agent.stream({"messages": [HumanMessage(content=question)]}):
    if "messages" in s:
        messages.append(s["messages"])

result = {"messages": [item for sublist

In [ ]:
# הדפסה מסודרת
import textwrap
line_width = 70

# הדפסת התשובה הסופית עם ירידות שורות ורוחב מוגבל
final_response_content = result["messages"][-1].content

if isinstance(final_response_content, list):
    for part in final_response_content:
        if isinstance(part, dict) and part.get("type") == "text":
            full_text = part.get("text", "")
            # Split the full text into paragraphs based on double newlines
            paragraphs = full_text.split('\n\n')
            for i, para in enumerate(paragraphs):
                wrapped_text = textwrap.fill(para, width=line_width)
                print(wrapped_text)
                if i < len(paragraphs) - 1: # Add a blank line between paragraphs, but not after the last one
                    print()
        else:
            # Fallback for unexpected part types in the list
            print(part)
elif isinstance(final_response_content, str):
    # If for some reason it's a single string, handle it similarly
    paragraphs = final_response_content.split('\n\n')
    for i, para in enumerate(paragraphs):
        wrapped_text = textwrap.fill(para, width=line_width)
        print(wrapped_text)
        if i < len(paragraphs) - 1:
            print()
else:
    # Fallback for unexpected content type
    print(final_response_content)